# Partial Differential equations: Finite differences
---
REFS:
- Franklin, Computational methods for Physics
- Landau y Paez, Computational physics
- Boudreau et al, Applied Computational Physics
---

## Why Partial Differential Equations?
Many physical systems depend simultaneously on **space** and **time**.

For example:

- The temperature inside a metal rod changes with position and time.
- The electric potential inside a region depends on spatial coordinates.
- Sound waves propagate through air.
- Quantum particles are described by wavefunctions that evolve in space and time.
- Fluids exhibit velocity and pressure fields varying throughout space.

Such systems cannot be described by ordinary differential equations (ODEs), because ODEs involve derivatives with respect to a single independent variable.

Instead, we need **partial differential equations (PDEs)**.

A PDE relates a function

\begin{equation}
u(x,y,z,t)
\end{equation}

to its partial derivatives with respect to several variables. For nice visualizations check [VisualPDE](https://visualpde.com/explore.html).

```{iframe} https://www.youtube.com/embed/ly4S0oi3Yz8?si=S5FlztSqkocawGO0
:align: center
:width: 100%
```



| Physical phenomenon | Governing PDE | Type |
|---|---|---|
| Electrostatics (charges at rest) | Poisson / Laplace equation | Elliptic |
| Heat conduction, particle diffusion | Heat / diffusion equation | Parabolic |
| Vibrating string, sound, light | Wave equation | Hyperbolic |
| [Fluid flow (incompressible)](https://paveldogreat.github.io/WebGL-Fluid-Simulation/) | Navier–Stokes equation | Mixed / nonlinear |
| Quantum mechanics | Schrödinger equation | Parabolic (in imaginary time) |
| Gravity (general relativity) | Einstein field equations | Hyperbolic |



```{tip} Laplace Equation
\begin{equation}
\nabla^2 \phi = 0
\end{equation}
Describes electrostatic potentials in charge-free regions.
```

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("deQAnSDPrSs", width=315, height=560)

```{tip} Heat Equation
\begin{equation}
\frac{\partial T}{\partial t} = D\nabla^2 T
\end{equation}
Describes diffusion of heat.
```

```{iframe} https://visualpde.com/sim/?preset=heatEquation1D
:align: center
:width: 90%
From visual pde
```

```{iframe} https://www.youtube.com/embed/uVKMY-WTrVo?si=igW34UBr8oNjdb9N
:align: center
:width: 100%
```

```{tip} Wave Equation
\begin{equation}
\frac{\partial^2 u}{\partial t^2} = c^2\nabla^2 u
\end{equation}
Describes vibrations, sound, and electromagnetic waves.
```


```{iframe} https://phet.colorado.edu/sims/html/wave-on-a-string/latest/wave-on-a-string_all.html
```

```{iframe} https://visualpde.com/sim/?preset=waveEquation1D
```

```{iframe} https://visualpde.com/sim/?preset=waveEquation3D
```

See more at <https://www.acs.psu.edu/drussell/demos.html>, <https://falstad.com/ripple/>

## Classification of PDEs

All linear second-order PDEs in two variables can be written as

\begin{equation}
A \frac{\partial^2 u}{\partial x^2} + B \frac{\partial^2 u}{\partial x \partial y} + C \frac{\partial^2 u}{\partial y^2} + \text{lower order terms} = 0.
\end{equation}

The sign of the *discriminant* $\Delta = B^2 - 4AC$ determines the character of the equation — and completely determines the type of numerical method needed:

```
Discriminant < 0  →  Elliptic   (e.g. Laplace, Poisson)
                      Describes steady states.
                      Boundary conditions on a closed domain.

Discriminant = 0  →  Parabolic  (e.g. Heat / diffusion)
                      Describes irreversible evolution in time.
                      Initial condition + boundary conditions.

Discriminant > 0  →  Hyperbolic (e.g. Wave equation)
                      Describes reversible wave propagation.
                      Initial conditions + boundary conditions.
```

This is not just a mathematical curiosity. The physics is completely different in each case:

- **Elliptic** equations have no preferred direction — a boundary perturbation is felt everywhere instantly. Perfect for equilibrium problems.
- **Parabolic** equations have a preferred time direction — information propagates at infinite speed but is damped. Perfect for diffusion.
- **Hyperbolic** equations have a *finite* propagation speed — information travels along *characteristics*. Perfect for wave phenomena.



## Why Numerical Methods?

Only a limited number of PDEs possess analytical solutions. Even simple geometries may prevent exact solutions. For simple geometries and boundary conditions, exact solutions exist and are taught in mathematical physics courses (separation of variables, Green's functions, Fourier series…). For example:

\begin{equation}
\nabla^2 V = 0, \quad V(x,0) = \sin(\pi x), \quad \text{zero on other walls}
\end{equation}
has the analytical solution $V(x, y) = \sin(\pi x)\, e^{-\pi y}$.

But reality rarely cooperates:

- Irregular geometries (a heart, a turbine blade, a microchip)
- Nonlinear material properties ($\alpha$ depending on temperature)
- Coupled systems (temperature affecting electric conductivity)
- Moving boundaries
- Heat diffusion in an irregular object.
- Electric potential in a complex device.
- Vibrations of a non-uniform membrane.
- Fluid flow around an airplane wing.

In all such cases **numerical methods** are the only practical tool.

The basic idea is simple:

1. Replace continuous space by a grid.
2. Replace derivatives by finite differences.
3. Solve the resulting algebraic equations.

This process converts a PDE into a problem that computers can handle.


## The Finite Difference Idea (in One Picture)

All the numerical methods in this chapter share one core idea: replace continuous derivatives by discrete differences on a grid.

For a function $u(x)$ sampled at points $x_i = x_0 + i\Delta x$:

\begin{equation}
\frac{\partial u}{\partial x}\bigg|_i \approx \frac{u_{i+1} - u_{i-1}}{2\Delta x} \qquad (\text{central difference, } O(\Delta x^2))
\end{equation}

\begin{equation}
\frac{\partial^2 u}{\partial x^2}\bigg|_i \approx \frac{u_{i+1} - 2u_i + u_{i-1}}{\Delta x^2} \qquad (\text{central second difference, } O(\Delta x^2))
\end{equation}

Substituting these into a PDE turns it into a system of **algebraic equations** — one per grid point — that a computer can solve.

The key parameters are:
- **Grid spacing** $\Delta x$, $\Delta y$, $\Delta t$: smaller → more accurate, but more computation.
- **Stability**: some methods require $\Delta t$ to be small relative to $\Delta x$ to avoid blow-up.
- **Boundary conditions**: what you fix at the edges completely determines the solution.

---



## The Poisson equation
The following is an example of an elliptical differential equation, determined by the boundary conditions :

\begin{equation}
\nabla^2 V = \left[ \frac{\partial^2 V}{\partial x^2} + \frac{\partial^2 V}{\partial y^2} + \frac{\partial^2 V}{\partial z^2}\right] = -\frac{\rho}{\epsilon_0}
\end{equation}

In 2D, there are several solutions and the actual one depends on the boundary conditions. For example, for the Laplace equation you have

\begin{align}
V(x, y) & = V_0,\\
V(x, y) & = \frac{1}{2} A (x^2 - y^2) + Bx + Cy + D
\end{align}
as solutions. We the density $\rho$ is null, we have the Laplace equation. It appears also when the heat equation stabilizes


A [finite difference](https://en.wikipedia.org/wiki/Finite_difference) method discretizes the derivatives on a grid. For instance, using the central difference it is possible to rewrite the 2D Poisson equation as ([Laplace equation](https://en.wikipedia.org/wiki/Laplace%27s_equation) corresponds to $\rho = 0$ )

\begin{equation}
\frac{V(x_i + \Delta x, y_j) - 2V(x_i, y_j) + V(x_i - \Delta x, y_j)}{\Delta x^2} +\frac{V(x_i, y_j + \Delta y) - 2V(x_i, y_j) + V(x_i, y_j - \Delta y)}{\Delta y^2} = -\frac{\rho(x_i, y_j)}{\epsilon_0}.
\end{equation}

which can be rewritten as  (let's assume $\Delta x = \Delta y = \Delta$)

\begin{equation}
\frac{V_{i+1, j} - 2V_{i, j} + V_{i-1, j}}{\Delta^2} + \frac{V_{i,j+1} - 2V_{i, j} + V_{i, j-1}}{\Delta^2} = -\frac{\rho_{i, j}}{\epsilon_0},
\end{equation}

and, therefore

\begin{equation}
V_{i+1, j} + V_{i-1, j} + V_{i,j+1} + V_{i, j-1} - 4V_{i, j} = -\frac{\rho_{i, j}\Delta^2}{\epsilon_0}.
\end{equation}

This can be seen in two ways: as a matrix equation or as an equation showing that the value at a given position, $V_{i, j}$, is kind of an average of the surrounding neighbors and the source term ([Jacobi method](https://en.wikipedia.org/wiki/Jacobi_method)), as seen (ref: https://barbagroup.github.io/essential_skills_RRC/laplace/1/#laplaces-equation)

```{figure} https://barbagroup.github.io/essential_skills_RRC/laplace/figures/laplace.svg
:width: 90%
:align: center
```




<!-- <img src="https://barbagroup.github.io/essential_skills_RRC/laplace/figures/laplace.svg" alt="adads" width="50%" align="center"> -->

<!-- <img src="https://barbagroup.github.io/essential_skills_RRC/laplace/figures/laplace.svg"> -->

The first approach can be solved by using the typical matrix methods (on the banded matrix obtained), while the second one gives raise to the [relaxation method](https://en.wikipedia.org/wiki/Relaxation_(iterative_method)), where you sweep the matrix several times until stabilization is achieved. 


## Relaxation solution
Let's solve the Laplace problem ($\rho = 0$) using both methods with a simple example. Fix the boundaries as

\begin{align}
V(x, 0) &= \sin(\pi x),\\
V(x, a) &= 0, \\
V(0, y) &= 0, \\
V(a, y) &= 0, \\
\end{align}

with $x, y \in [0, 1]$, and $a $ any value on the border. These are called [Dirichlet boundary conditions](https://en.wikipedia.org/wiki/Dirichlet_boundary_condition) (where you specify the function values at the boundary). Finally, use at least $N_x = N_y = 25$, so $\Delta = 1/25$, $x_i = x_0 + i\Delta, y_j = y_0 + j\Delta$. We need to solve the equation

\begin{equation}
V_{i, j} = \frac{V_{i+1, j} + V_{i-1, j} + V_{i,j+1} + V_{i, j-1}}{4}.
\end{equation}

To do so, we apply this relationship to each point in the grid, without overwriting the boundary conditions, and we do that until the values are not changing much. Firs, let's define the problem conditions (to be changed later)

In [ ]:
from IPython.display import YouTubeVideo
YouTubeVideo("deQAnSDPrSs", width=315, height=560)

To solve the system, we will create functions to
- Define the problem domain
- Apply the boundary conditions
- Perform a relaxation iteration: this can de done in several ways
- Actually solve the system, calling the appropriate iteration and comparing

### Problem setup

In [ ]:
# Global problem definition
import numpy as np
def setup_problem(N):
    XMIN = 0.0
    YMIN = 0.0
    DELTA = 1.0/N # L = 1
    X = XMIN + DELTA*np.arange(0, N)
    Y = XMIN + DELTA*np.arange(0, N)
    return X, Y

## Boundary conditions 
Now create a function for the boundary conditions

In [ ]:
# Boundary conditions
def bc(matrix, x, y):
    # YOUR CODE HERE
    pass

In [ ]:
def is_boundary(ix, iy, N):
    "Returns true if a point is a boundary. Assumes square grid"
    if ix == 0: return True
    if iy == 0: return True
    if ix == N-1: return True
    if iy == N-1: return True
    return False
    

In [ ]:
# Visualize the boundary condition
N = 25
X, Y = setup_problem(N)
M = np.zeros((N, N))
bc(M, X, Y)

import matplotlib.pyplot as plt
img = plt.imshow(M)
plt.colorbar(img)
plt.tight_layout()
#plt.savefig("fig/laplace_boundary.png", dpi=120)

```{figure} fig/laplace_boundary.png
:alt: 
:class: bg-primary
:width: 70%
:align: center
```


## Iteration

```{figure} https://aquaulb.github.io/book_solving_pde_mooc/_images/GSgrid_e.png
:alt: 
:class: bg-primary
:width: 90%
:align: center
```

And this will be a simple example for single iteration step. In this case, we are avoiding the external walls since they are boundary conditions. We are using the fast numpy notation (see also https://becksteinlab.physics.asu.edu/learning/76/phy494-computational-physics?wchannelid=bp7lsp6dmx&wmediaid=284owsl3dp), but of course you can use loops (specially for more complex boundary problems). Here we are updating into the same matrix. 

In [ ]:
# Full iteration using for loops
def jacobiITER(matrix):
    # YOUR CODE HERE
    pass

In [ ]:
# Test
# Visualize the boundary condition
N = 25
X, Y = setup_problem(N)
M = np.zeros((N, N))
bc(M, X, Y)
jacobiITER(M)

import matplotlib.pyplot as plt
img = plt.imshow(M)
plt.colorbar(img)
plt.tight_layout()
#plt.savefig("fig/laplace_1iter.png", dpi=120)

```{figure} fig/laplace_1iter.png
:alt: 
:class: bg-primary
:width: 80%
:align: center
```


## System solution

Finally, let's perform some iterations to check if the system is converging somewhere

In [ ]:
import time
def solve_system(N, niter, iter_method):
    X, Y = setup_problem(N)
    V = np.zeros((N, N))
    bc(V, X, Y)
    # Example iteration
    for step in range(niter):
        #print(V)
        t0 = time.time()
        iter_method(V)
        t1 = time.time()
        print(f"Total time iterating: {t1-t0}")
        #print(V)
        #print()

In [ ]:
solve_system(N=500, niter=10, iter_method=jacobiITER)


Notice that we can also use numpy directly as


In [ ]:
# Full iteration using only numpy indexing, but implicitly assumes the boundary condition
def jacobi_np(matrix):
    """
    This uses numpy notation to perform the vectorized update
    """
    matrix[1:-1, 1:-1] = 0.25*(matrix[2:, 1:-1] + matrix[0:-2, 1:-1] + 
                               matrix[1:-1, 2:] + matrix[1:-1, 0:-2])


In [ ]:
solve_system(N=500, niter=10, iter_method=jacobi_np)

But the numpy approach can be complicated to write if the boundary conditions are not easy to express.

### Graphical verification
Let's check it  graphically

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
def solve_system(N, niter, iter_method, NDISPLAY=0):
    # figs setup
    if NDISPLAY > 0:
        NCOLS = 4
        NROWS = int(((niter/NDISPLAY)/NCOLS))
        if ((niter/NDISPLAY)%NCOLS) != 0:
            NROWS += 1
        fig, ax = plt.subplots(NROWS, NCOLS, sharex=True, sharey=True, figsize=(15, 15))
        ax = ax.flatten()
    # Problem setup
    X, Y = setup_problem(N)
    V = np.zeros((N, N))
    bc(V, X, Y)

    # aux vars in case the iteration returns something
    aux = np.zeros(niter)
    
    # Example iteration
    idisplay= 0
    for step in range(1, niter+1):
        if NDISPLAY>0 and (step%NDISPLAY == 0 or step==1):
            # ax[idisplay].set_title(step)
            # ax[idisplay].imshow(V)
            ax[idisplay].set_aspect('equal', 'box')
            ax[idisplay].pcolormesh(V)
            ax[idisplay].set_title(f"step={step}/{niter}")
            CS = ax[idisplay].contour(V, cmap='Blues')
            ax[idisplay].clabel(CS, CS.levels, inline=True)
            idisplay += 1
        aux[step-1] = iter_method(V)
    if NDISPLAY > 0:
        plt.tight_layout()
        #fig.savefig("fig/laplace_relax.png", dpi=150)
    return aux

In [ ]:
tmp = solve_system(N=20, niter=200, iter_method=jacobiITER, NDISPLAY=20)

```{figure} fig/laplace_relax.png
:alt: 
:class: bg-primary
:width: 90%
:align: center
```


To have a quantitative relaxation index, we could compare the new values with the old ones, so let's redefine the iteration step method to return the estimator 

In [ ]:
def jacobi_diff(matrix):
    matrix_old = matrix.copy() # Might be slow, creates copies all the time
    # YOUR CODE HERE
    pass


Notice that the convergence depends strongly on the space resolution:

In [ ]:
for N in [5, 7, 10, 20, 50 , 100, 200, 300]:
    print(N)
    diff = solve_system(N=N, niter=1000, iter_method=jacobi_diff, NDISPLAY=0)
    plt.plot(diff, label=f"{N=}")
plt.xscale("log")
plt.yscale("log")
plt.legend()
#plt.savefig("fig/laplace_relax_2.png", dpi=120)

```{figure} fig/laplace_relax_2.png
:alt: 
:class: bg-primary
:width: 90%
:align: center
```


## A performance discussion 

But this loop version is very slow. When you need to optimize something, you need to measure it (using profilers). Here we will use https://jakevdp.github.io/PythonDataScienceHandbook/01.07-timing-and-profiling.html  and https://mortada.net/easily-profile-python-code-in-jupyter.html . 

In [ ]:
%%timeit
tmp = solve_system(N=50, niter=100, iter_method=jacobiITER)
# 87.5 ms ± 1.5 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)

In [ ]:
%%timeit
tmp = solve_system(N=50, niter=100, iter_method=jacobi_np)
# 652 μs ± 10.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

We can optimize even more with `numba` (see https://stackoverflow.com/questions/61161072/optimizing-vectorized-operations-made-by-sections-in-numpy): 

In [ ]:
#!conda install -y numba
#!pip install -y numba
!uv pip install numba numpy==2.0


In [ ]:
from numba import jit, njit
import numpy as np
@njit(parallel=False)
def jacobi_opt(matrix):
    #matrix_old = matrix.copy() # Might be slow, creates copies all the time
    N = matrix.shape[0]  # (N, N)
    # Implement here the full jacobi iteration again without calling any other function
    # So boundary contition must be here
    # YOUR CODE HERE
    pass
    #rij = matrix - matrix_old
    #diff = np.linalg.norm(rij)
    return 0.0

In [ ]:
%%timeit 
tmp = solve_system(N=50, niter=100, iter_method=jacobi_opt)
# 457 μs ± 558 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Exercise: Plane capacitor
Now , besides the given boundary conditions, add two lines representing a plane capacitor, with opposite voltages of $\pm 3$ volts, with length $2L_y/3$, at $L_x/3$ and $2L_x/3$. Plot the final solution as a 2D map and as a 3D map. 

In [ ]:
# YOUR CODE HERE
pass


## Exercise: circular ring
Now , besides the given boundary conditions, add a circular ring with constant voltage of 3 volts and radius $L_x/4$. Plot the final solution as a 2D map and as a 3D map. 

In [ ]:
# YOUR CODE HERE
pass


## Exercise: Plot the electrical field
For any of the previous examples, use (for example) a quiver plot to plot the electric field, defined as
\begin{equation}
\vec E = -\vec\nabla V
\end{equation}
See https://nbviewer.org/urls/www.numfys.net/media/notebooks/relaxation_methods.ipynb

In [ ]:
# YOUR CODE HERE
pass


## Exercise: Add two square charges
Solve the Poisson equation with two square charges of size $L_x/10$ on the diagonal. Use $\epsilon_0 = 1$

In [ ]:
# YOUR CODE HERE
pass


## Exercise: Implement over-relaxation
Calibrate $\omega$  (see https://en.wikipedia.org/wiki/Successive_over-relaxation)

In [ ]:
from numba import jit
@jit
def jacobi_sor_opt(matrix, omega):
    matrix_old = matrix.copy() # Might be slow, creates copies all the time
    # YOUR CODE HERE
    pass
    rij = matrix - matrix_old
    diff = np.linalg.norm(rij)
    return diff
    #return gs_opt(matrix)

In [ ]:
#for N in [5, 7, 10, 20, 50 , 100, 200, 500]:
for N in [5]:
    print(N)
    OMEGA=1.9
    diff = solve_system(N=N, niter=1000, iter_method=lambda m: jacobi_sor_opt(m, OMEGA))
    plt.plot(diff, label=f"{N=}")
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.savefig("fig/laplace_jacobi_sor.png")

```{figure} fig/laplace_jacobi_sor.png
:alt: 
:class: bg-primary
:width: 80%
:align: center
```


## Exercise: Stop after tolerance is achieved
Modify the iteration  procedure to stop after some given precision has been achieved

In [ ]:
# YOUR CODE HERE
pass
